# TinyLM quickstart

This notebook walks through the core `tinylm` API:

1. Build a Transformer language model from a typed `ModelConfig`.
2. Run a forward pass on random token ids.
3. Train a tiny BPE tokenizer and round-trip a string.

For real training and checkpoint-based generation, see `docs/running.md`.

In [ ]:
import torch

from tinylm import ModelConfig

torch.manual_seed(0)

config = ModelConfig(
    d_model=128,
    num_heads=4,
    d_ff=512,
    vocab_size=512,
    num_layers=2,
    max_seq_len=128,
)
model = config.build(device="cpu")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
tokens = torch.randint(0, config.vocab_size, (2, 32))
logits = model(tokens)
print("input: ", tuple(tokens.shape))
print("logits:", tuple(logits.shape))

## Byte-level BPE tokenizer

Train a small BPE vocabulary on a fixture corpus, then encode and decode text.

In [ ]:
from tinylm import Tokenizer, run_bpe_train

vocab, merges = run_bpe_train(
    "../tests/fixtures/corpus.en",
    vocab_size=300,
    special_tokens=["<|endoftext|>"],
)
tokenizer = Tokenizer(vocab, merges, ["<|endoftext|>"])

text = "Hello, world! This is TinyLM."
ids = tokenizer.encode(text)
print("ids:    ", ids)
print("decoded:", repr(tokenizer.decode(ids)))
assert tokenizer.decode(ids) == text